# sci_ai_engine — Phase A/B training on an AMD ROCm developer-cloud notebook

Adapted from `colab_notebook.ipynb`. Key differences from the Colab version:
- No Google Drive — this platform gives a local persistent-ish disk instead.
  **Persistence is unverified**; cell 2 below writes a dated marker file so
  you can tell after a restart whether storage actually survived. Until you
  confirm it does, treat long unattended runs as risky and check in often.
- GPU is AMD (ROCm), not CUDA — but PyTorch's `torch.cuda` API works
  transparently on ROCm builds, so `train.py`/`finetune.py` need no changes.
- No `ANTHROPIC_API_KEY` needed anywhere — Phase B uses `synth_dataset.py`.

## 1. Verify GPU + check whether local storage persists across sessions

In [ ]:
import torch, os, subprocess
import datetime as _dt  # aliased: a bare `datetime` global here can collide with
                         # this platform's own kernel-side tooling (observed: it
                         # broke this platform's progress-reporting hook, which
                         # expected `datetime` to already be bound to the class)

def run(cmd):
    """Uses subprocess directly rather than get_ipython().system(): the
    latter's return value isn't reliably an int across IPython/Jupyter kernel
    implementations (observed returning None here regardless of outcome,
    which made an exit-code check meaningless). subprocess.run's returncode
    is deterministic."""
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise SystemExit(f'Command failed (exit {result.returncode}): {cmd}')

print('torch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('VRAM (GB):', round(props.total_memory / 1e9, 1))
    print('bf16 supported:', torch.cuda.is_bf16_supported())

WORK_ROOT = '/workspace/sci_ai_engine'  # change if your platform's persistent mount differs
os.makedirs(WORK_ROOT, exist_ok=True)

marker = f'{WORK_ROOT}/.persistence_check'
if os.path.exists(marker):
    print('PERSISTENCE CONFIRMED — marker survived from a previous session:')
    print('  ', open(marker).read())
else:
    print('No marker found (first run, OR storage did not persist — rerun this cell')
    print('after a session restart to tell which one).')
timestamp = _dt.datetime.now(_dt.timezone.utc).isoformat()
with open(marker, 'w') as f:
    f.write('written at ' + timestamp + '\n')

## 2. Get the code (clone once; pulls latest on re-run)

In [ ]:
REPO_DIR = f'{WORK_ROOT}/sci_ai_engine'
if not os.path.exists(REPO_DIR):
    run(f'git clone https://github.com/sa9446/sci_ai.git {REPO_DIR}')
else:
    run(f'cd {REPO_DIR} && git pull')

## 3. Install training dependencies

Skips reinstalling `torch` if a GPU-capable build is already present —
the plain `torch>=2.3.0` line in `requirements-train.txt` would otherwise
resolve to a CUDA/CPU wheel from PyPI and clobber this platform's ROCm build.

In [ ]:
req_path = f'{REPO_DIR}/model_training/requirements-train.txt'
if torch.cuda.is_available():
    print('GPU-capable torch already present — installing remaining deps only.')
    lines = [l for l in open(req_path) if not l.strip().lower().startswith('torch')]
    tmp_req = f'{WORK_ROOT}/requirements-train-notorch.txt'
    with open(tmp_req, 'w') as f:
        f.writelines(lines)
    run(f'pip install -q -r {tmp_req}')
else:
    run(f'pip install -q -r {req_path}')

## 4. One-time corpus prep (skip once train.bin/val.bin already exist)

The default fetch size (2000 arXiv abstracts/category, 300 Wikipedia
pages/category) turned out too small — it caused Phase A to badly overfit
by iteration ~500 (train loss collapsed to 0.05 while val loss never
improved past its iter-250 value). `--arxiv-max-per-category`/
`--wikipedia-max-pages-per-category` in the corpus-prep cell below are set
much higher for a real run.

If you already ran the old cell with the too-small corpus, run the cleanup
cell immediately below first — it wipes the undersized
data/tokenizer/Phase-A-checkpoint so the corpus-prep cell actually refetches
instead of seeing `train.bin` already exists and skipping.

In [ ]:
import shutil

# One-time migration cleanup: wipes the corpus/tokenizer/Phase-A checkpoint
# produced by the old too-small fetch_corpus.py defaults. Leave this False on
# any later rerun of this notebook (e.g. after a restart) — Phase A/B are
# meant to --resume from real progress, not get wiped every run.
CLEAN_STALE_CORPUS = True

if CLEAN_STALE_CORPUS:
    for _path in [
        f'{WORK_ROOT}/data/raw', f'{WORK_ROOT}/data/train.bin', f'{WORK_ROOT}/data/val.bin',
        f'{WORK_ROOT}/tokenizer', f'{WORK_ROOT}/checkpoints/phaseA',
    ]:
        if os.path.isdir(_path):
            shutil.rmtree(_path)
            print('removed dir:', _path)
        elif os.path.exists(_path):
            os.remove(_path)
            print('removed file:', _path)
        else:
            print('not present:', _path)
else:
    print('CLEAN_STALE_CORPUS is False — skipping (expected on normal reruns).')

In [ ]:
DATA_DIR = f'{WORK_ROOT}/data'
if not os.path.exists(f'{DATA_DIR}/train.bin'):
    run(
        f'cd {REPO_DIR}/model_training && python data/fetch_corpus.py --out-dir {DATA_DIR}/raw '
        f'--arxiv-max-per-category 20000 --wikipedia-max-pages-per-category 3000'
    )
    run(f'cd {REPO_DIR}/model_training && python tokenizer/train_tokenizer.py --corpus-dir {DATA_DIR}/raw --out-dir {WORK_ROOT}/tokenizer')
    run(f'cd {REPO_DIR}/model_training && python data/prepare.py --raw-dir {DATA_DIR}/raw --tokenizer-path {WORK_ROOT}/tokenizer/tokenizer.json --out-dir {DATA_DIR}')
else:
    print('train.bin/val.bin already present — skipping corpus prep.')

## 5. Phase A pretraining (resumable — safe to re-run after any restart)

This platform reported ~205 GB VRAM (MI300X-class), far more than the T4's
~14.56 GiB that OOM'd at `--batch-size 12` (see `colab_notebook.ipynb`).
`--batch-size 32` here is still conservative headroom, not a limit — raise it
if you want faster wall-clock training; effective batch size stays
`batch-size * grad-accum-steps` either way.

In [ ]:
import json, os

tokenizer_path = f'{WORK_ROOT}/tokenizer/tokenizer.json'
if not os.path.exists(tokenizer_path):
    raise SystemExit(
        f'{tokenizer_path} not found — run the cleanup cell and cell 4 (corpus prep) '
        'first and let them finish before running Phase A.'
    )

with open(tokenizer_path) as f:
    vocab_size = len(json.load(f)['model']['vocab'])
print('vocab_size =', vocab_size)

run(
    f'cd {REPO_DIR}/model_training && python train.py '
    f'--data-dir {DATA_DIR} --out-dir {WORK_ROOT}/checkpoints/phaseA --resume '
    f'--vocab-size {vocab_size} --max-iters 50000 --batch-size 32 --grad-accum-steps 3 '
    f'--eval-interval 250 --save-interval 250'
)

## 6. Synthetic Phase B dataset (no Anthropic dependency — hand-written physics formulas)

In [ ]:
run(
    f'cd {REPO_DIR}/model_training && python synth_dataset.py '
    f'--out-path {WORK_ROOT}/data/distilled.jsonl --num-queries 2000'
)

## 7. Phase B fine-tune (resumable, same pattern as Phase A)

In [ ]:
run(
    f'cd {REPO_DIR}/model_training && python finetune.py '
    f'--base-checkpoint {WORK_ROOT}/checkpoints/phaseA/ckpt_best.pt '
    f'--dataset-path {WORK_ROOT}/data/distilled.jsonl '
    f'--out-dir {WORK_ROOT}/checkpoints/phaseB --resume'
)